# A tour of the rebuilt ProbPipe API

This notebook walks through the API on `dev/overhaul`, with one hierarchical model as the thread: the non-centered eight-schools model, whose parameters form the nested record `{population: {mu, tau}, groups: {theta_tilde}}`.
Each markdown cell says what the next cell shows, and each output is the library's own repr, declaration, table, or error message.
The last section lists what felt awkward while writing the notebook.

## Status

The design reference in `design/` is the target, and `dev/overhaul` implements most of it.
The pieces below are transitional, and each gets a one-line note where it first appears.

- **Imports.** The new operations are imported from their modules, as `tests/_ops.py` does. `probpipe.sample`, `probpipe.mean`, and the other top-level names are still the earlier implementations in `core/ops.py`. The next wave switches the exports, and the imports become `from probpipe import ...`.
- **Seeds.** `key=` arguments retire, and reproducible draws come from `workflow_run(seed=...)` (S1).
- **`expectation`.** It resolves through its own method registry today. It becomes the derived operation `mean(evaluate(f, d))`, so its controls and method names become `evaluate`'s and the registry retires (Z8). Every functional is an operation (Z10).
- **Labels.** A field view, a selection, and a marginal keep their parent's label, so `prior["beta"]` is labeled `prior` (Z5). The code still labels them by their path, and a fix is in progress.
- **Wave-2 review fixes in progress.** A cell that meets one of these keeps the behavior and marks it **Known bug**:
  1. two accesses of one batch element do not co-sample;
  2. an SG-MCMC or elliptical-slice chain on a factored prior starts outside the support;
  3. `probpipe.condition_on` of a factored joint at an upstream field runs MCMC where the exact slice applies;
  4. the empirical enumeration's exactness depends on argument order;
  5. `d[path]` at a regrouped node raises;
  6. `evaluate.with_options(method=...)` cannot name a rule;
  7. the moment matcher's `check` and its call disagree on dtypes;
  8. `from_distribution(..., check_support=False)` raises;
  9. `check` lists capability routes as approximate;
  10. a selection's marginal orders its fields by factor.
- **Open design questions**, marked **Open question** where they arise:
  1. the predictive checks take a sampling kernel, after which `GenerativeLikelihood` retires;
  2. a public capability for the density of a subset of observations;
  3. the top-level exports of `glm_likelihood` and `GaussianProcess`.
- **Docs.** The other notebooks in `docs/` are not updated yet, and a later pass rewrites them.

A cell marked **Finding** shows a behavior that differs from `design/` or surprised me and is not on the lists above.

## Setup

The operations come from their modules until the exports switch.
`show_report` prints a `CallReport` with one route per line.

In [1]:
import tempfile
import warnings
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import probpipe
from probpipe import (
    BijectorTransformedDistribution, BootstrapDistribution, BootstrapReplicateDistribution,
    Dirichlet, DistributionBatch, EmpiricalDistribution, Gamma, HalfCauchy, KDEDistribution,
    MathematicalDomainError, MultivariateNormal, Normal, NumericArray, NumericArrayBatch,
    NumericArraySpec, NumericRecordBatch, Opaque, OutputSpec, Poisson, PyMCModel, Record,
    RecordBatch, RecordSpec, ResolutionError, StanModel, StudentT, SupportsCovariance,
    SupportsLogProb, SupportsMean, SupportsQuantile, SupportsSampling, SupportsUnnormalizedLogProb,
    SupportsVariance, bijector_for, converter_registry, function, iterate, positive,
    provenance_ancestors, real, replay_run, workflow_run,
)
from probpipe.distributions import (
    ConditionalDistribution, SupportsConditionalLogProb, SupportsConditionalSampling,
    SupportsFactors, SupportsMarginals,
)
from probpipe.families import PoissonFamily, glm_likelihood
from probpipe.operations import operation_registry

# The operations, from their modules until the top-level exports switch.
from probpipe.operations._condition import condition_on
from probpipe.operations._convert import convert
from probpipe.operations._density import log_prob, unnormalized_log_prob
from probpipe.operations._evaluate import evaluate
from probpipe.operations._marginal import factor, marginal
from probpipe.operations._moments import cov, expectation, mean, quantile, variance
from probpipe.operations._sample import sample

warnings.filterwarnings("ignore", message=r"Explicitly requested dtype float64")
warnings.filterwarnings("ignore", message=r"Loading a shared object")


def show_report(report):
    for route in report.routes:
        status = {True: "feasible", False: "declined", None: "unresolved"}[route.feasible]
        print(f"  {route.method_name:<34} {status:<9} exact={route.exact}  {route.description}")
    print("selected:", report.selected.method_name, "| exact:", report.selected.exact)

## 1. Values

A value is a tracked term: a representation together with a name and a spec.
`NumericArray` holds one array, whose shape is the event shape.

In [2]:
effect = NumericArray("effect", jnp.array([28.0, 8.0, -3.0]))
print(repr(effect))
print(effect.spec)
print(repr(effect + 1.0))

NumericArray('effect', Array([28.,  8., -3.], dtype=float32))
NumericArraySpec(shape=(3,), dtype=dtype('float32'), support=None)
Array([29.,  9., -2.], dtype=float32)


**Finding:** arithmetic on a `NumericArray` returns a bare `jax.Array`, while design III.1 returns a tracked term under a derived name.

A `Record` is a named tree of values, and a mapping-valued field becomes a subtree.
A subtree whose leaves are all numeric is promoted to a `NumericRecord`, so a school's record holds a numeric `data` subtree beside a string label.

In [3]:
school = Record("school_A", {"data": {"effect": 28.0, "se": 15.0}, "label": "A"})
print(type(school).__name__, school.spec)
print(list(school.keys()))
data = school.at_path("data")
print(type(data).__name__, data.name, data.spec)
print(repr(school["data/effect"]), repr(school["label"]))

Record RecordSpec(data=NumericRecordSpec(effect=(), se=()), label=None)
['data/effect', 'data/se', 'label']
NumericRecord data NumericRecordSpec(effect=(), se=())
Array(28., dtype=float32, weak_type=True) 'A'


**Finding:** `record[key]` returns the raw leaf, a `jax.Array` or the string, where design III.5 returns a view of the field's kind.

The eight-schools parameters form a `NumericRecord`.
It has the flat-vector interface, and `map` applies an elementwise function while keeping the structure.

In [4]:
theta = Record(
    "theta",
    {"population": {"mu": 4.0, "tau": 3.0}, "groups": {"theta_tilde": jnp.linspace(-1.0, 1.0, 8)}},
)
print(type(theta).__name__, theta.spec)
print(theta.vector_size, theta.to_vector())
print(repr(theta.map(jnp.abs)))
try:
    theta + theta
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

NumericRecord NumericRecordSpec(population=NumericRecordSpec(mu=(), tau=()), groups=NumericRecordSpec(theta_tilde=(8,)))
10 [ 4.          3.         -1.         -0.71428573 -0.42857143 -0.14285709
  0.1428572   0.42857146  0.71428585  1.        ]
NumericRecord(population=NumericRecord(mu=Array(4., dtype=float32, weak_type=True), tau=Array(3., dtype=float32, weak_type=True)), groups=NumericRecord(theta_tilde=array(shape=(8,))))
TypeError: unsupported operand type(s) for +: 'NumericRecord' and 'NumericRecord'


**Finding:** `+` between two records of one schema raises `TypeError`, while design III.5 gives `NumericRecord` the vector-space operators `+`, `-`, and scalar `*` and `/`.

`Opaque` wraps a value that no other kind admits, such as a tuple.
A mapping is refused, since the value layer reads a mapping as a subtree.

In [5]:
note = Opaque("source", ("Rubin (1981)", "SAT points"))
print(repr(note), note.spec, note.value)
try:
    Opaque("source", {"study": "Rubin (1981)"})
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

Opaque('source', ('Rubin (1981)', 'SAT points')) OpaqueSpec(meta=None) ('Rubin (1981)', 'SAT points')
TypeError: Opaque holds one unstructured value, and the value layer reads a mapping as a subtree rather than a leaf; wrap it as a Record, or as a non-mapping value


A `RecordBatch` stores records column by column on named levels.
Numeric and non-numeric fields sit side by side: a position gives a record, and a field gives its column.

In [6]:
y_obs = jnp.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])
sigma = jnp.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])
J = y_obs.shape[0]
schools_data = RecordBatch.stack(
    [Record("school", {"effect": y, "se": s, "label": label}) for y, s, label in zip(y_obs, sigma, "ABCDEFGH")],
    level_name="school",
    name="schools",
)
print(repr(schools_data), schools_data.element_spec)
print(repr(schools_data[2]), schools_data[2].name)
print(repr(schools_data.at_levels(school=2)))
print(repr(schools_data["label"]))
print(repr(schools_data["effect"]))

RecordBatch(name='schools', school=8) RecordSpec(effect=(), se=(), label=None)
Record(effect=Array(-3., dtype=float32), se=Array(16., dtype=float32), label='C') schools[school=2]
Record(effect=Array(-3., dtype=float32), se=Array(16., dtype=float32), label='C')
OpaqueBatch(name="schools['label']", school=8)
Array([28.,  8., -3.,  7., -1.,  1., 18., 12.], dtype=float32)


**Finding:** the numeric column comes back as a bare `jax.Array`, while the non-numeric column is a tracked `OpaqueBatch`. Design III.6 returns a tracked batch column for both.

The raw form is one call away for a law, whose `raw()` is its backend object.
A value's representation is read through `.value`, `.values`, `to_nested_dict()`, or indexing, and `sample.with_options(raw=True)` returns raw draws (Section 4).

In [7]:
print(effect.value)
print(school.to_nested_dict())
print(Normal("x", 0.0, 1.0).raw())
print("raw() on NumericArray, Record, RecordBatch:", hasattr(effect, "raw"), hasattr(school, "raw"), hasattr(schools_data, "raw"))

[28.  8. -3.]
{'data': {'effect': Array(28., dtype=float32, weak_type=True), 'se': Array(15., dtype=float32, weak_type=True)}, 'label': 'A'}
tfp.distributions.Normal("Normal", batch_shape=[], event_shape=[], dtype=float32)
raw() on NumericArray, Record, RecordBatch: False False False


**Finding:** the value kinds and the batches have no `raw()`, while design II.4 makes `raw()` the single access point of every tracked term.

## 2. Naming

A label names an object and has no mathematical meaning, while a component name addresses a part of what the object produces (C5).
A family's component defaults to its label, and `event_spec` names another.

In [8]:
prior_beta = Normal("prior", 0.0, 1.0, event_spec=OutputSpec(beta=None))
print(prior_beta.name, list(prior_beta.event_spec.components))
renamed = prior_beta.with_name("slope_prior")
print(renamed.name, list(renamed.event_spec.components))
print(prior_beta["beta"] is prior_beta)
print(prior_beta.event_spec)

prior ['beta']
slope_prior ['beta']
True
OutputSpec(_component_name='beta', _term_spec=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real))


`with_name` changes the label alone, and indexing a whole-term law by its component returns the law itself.
The `OutputSpec` repr shows its private fields.

`with_path_names` renames nodes by their exact paths.
A target under the same parent renames in place, and a target under another parent moves the node, which promotes or demotes it.
All renames apply at once, so a swap is one call.

In [9]:
print("in place: ", list(school.with_path_names({"data/effect": "data/y"}).keys()))
print("promotion:", list(school.with_path_names({"data/effect": "effect"}).keys()))
print("demotion: ", list(school.with_path_names({"label": "meta/label"}).keys()))
print("swap:     ", school.with_path_names({"data/effect": "data/se", "data/se": "data/effect"}).to_nested_dict()["data"])
try:
    school.with_path_names({"effect": "y"})
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

in place:  ['data/y', 'data/se', 'label']
promotion: ['data/se', 'label', 'effect']
demotion:  ['data/effect', 'data/se', 'meta/label']
swap:      {'se': Array(28., dtype=float32, weak_type=True), 'effect': Array(15., dtype=float32, weak_type=True)}
KeyError: 'effect'


A key must be the exact path of a node, so the bare `effect` is refused; the `KeyError` names the key alone.

The same call renames a law's event.
Demoting the fields of a joint of three parametric priors builds the eight-schools prior with its nested record.
Demoting the component of a whole-term law is refused, since it would change the draw's kind.

In [10]:
flat_prior = Normal("mu", 0.0, 5.0) * HalfCauchy("tau", 0.0, 5.0) * Normal("theta_tilde", jnp.zeros(J), 1.0)
NEST = {"mu": "population/mu", "tau": "population/tau", "theta_tilde": "groups/theta_tilde"}
prior = flat_prior.with_path_names(NEST)
print(repr(flat_prior), list(flat_prior.event_spec.components))
print(repr(prior), list(prior.event_spec.components))
print(prior.event_spec.spec)
print("SupportsFactors:", isinstance(flat_prior, SupportsFactors), "->", isinstance(prior, SupportsFactors))
try:
    Normal("mu", 0.0, 5.0).with_path_names({"mu": "population/mu"})
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

FactoredDistribution(name='mu·tau·theta_tilde')

 ['mu', 'tau', 'theta_tilde']
_RenamedDistribution(name='mu·tau·theta_tilde', parent='mu·tau·theta_tilde') ['population', 'groups']
NumericRecordSpec(population=NumericRecordSpec(mu=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), tau=NumericArraySpec(shape=(), dtype=dtype('float32'), support=greater_than(0.0))), groups=NumericRecordSpec(theta_tilde=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)))
SupportsFactors: True -> False
ValueError: with_path_names() keeps the packaging, so the whole term's component 'mu' is renamed in place, not moved to 'population/mu'


**Finding:** the renamed joint is a private `_RenamedDistribution`, which no longer claims `SupportsFactors`. Design IV.1 renames a joint through its factors and returns a factored joint, and the docstring of `Distribution.with_path_names` promises `NotImplementedError` here (`probpipe/distributions/_distribution.py:496`). Sections 5 and 6 show the consequences.

## 3. Distributions

Each family derives its event declaration from its parameters: the component, shape, dtype, and support.
Capabilities are protocols, so `isinstance(d, SupportsMean)` reports whether a law implements a mean.

In [11]:
families = [
    Normal("mu", 0.0, 5.0),
    HalfCauchy("tau", 0.0, 5.0),
    Poisson("count", 3.0),
    Dirichlet("weights", jnp.ones(3)),
    MultivariateNormal("z", jnp.zeros(2), cov=jnp.array([[1.0, 0.5], [0.5, 2.0]])),
]
protocols = [SupportsSampling, SupportsLogProb, SupportsMean, SupportsVariance, SupportsCovariance, SupportsQuantile, SupportsMarginals]
pd.DataFrame(
    {
        "component": [next(iter(d.event_spec.components)) for d in families],
        "shape": [d.event_shape for d in families],
        "dtype": [str(d.dtype) for d in families],
        "support": [str(d.support) for d in families],
        **{p.__name__.removeprefix("Supports"): [isinstance(d, p) for d in families] for p in protocols},
    },
    index=[type(d).__name__ for d in families],
)

,component,shape,dtype,support,Sampling,LogProb,Mean,Variance,Covariance,Quantile,Marginals
Normal,mu,(),float32,real,True,True,True,True,True,True,False
HalfCauchy,tau,(),float32,greater_than(0.0),True,True,True,True,True,True,False
Poisson,count,(),float32,non_negative_integer,True,True,True,True,True,False,False
Dirichlet,weights,"(3,)",float32,simplex,True,True,True,True,True,False,False
MultivariateNormal,z,"(2,)",float32,real,True,True,True,True,True,True,False


A guard narrows a claim per instance.
`StudentT` claims `SupportsMean`, its guard decides existence once the degrees of freedom are concrete, and a mean known not to exist raises `MathematicalDomainError` rather than `ResolutionError` (II.7).
`HalfCauchy` claims a mean on the same terms.
Section 6 shows a guard that declines.

In [12]:
t = StudentT("t", 0.8, 0.0, 1.0)
print(isinstance(t, SupportsMean))
for law in (t, HalfCauchy("tau", 0.0, 5.0)):
    try:
        mean(law)
    except MathematicalDomainError as error:
        print(f"{type(error).__name__}: {error}")

True
MathematicalDomainError: the mean of 't' does not exist: it is finite only for degrees of freedom above one
MathematicalDomainError: the mean of 'tau' does not exist: E[X] is infinite


Parameters with more axes than one law needs give one law whose extra axes are event axes.
A vector `loc` gives independent coordinates, and a `MultivariateNormal` whose `loc` has shape `(4, 2)` gives four independent rows, so `log_prob` of one draw is a scalar in both cases.
Separate laws form a `DistributionBatch` (Section 8).

In [13]:
coords = Normal("theta_tilde", jnp.zeros(J), 1.0)
rows = MultivariateNormal("rows", jnp.zeros((4, 2)), cov=jnp.array([[1.0, 0.8], [0.8, 1.0]]))
print(repr(coords), coords.event_spec.spec)
print(repr(rows), rows.event_spec.spec)
with workflow_run(seed=0):
    draw = sample(rows)
print(draw.value)
print(repr(log_prob(coords, jnp.zeros(J))), repr(log_prob(rows, draw)))

Normal(name='theta_tilde', event_shape=(8,)) NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)
MultivariateNormal(name='rows', event_shape=(4, 2)) NumericArraySpec(shape=(4, 2), dtype=dtype('float32'), support=real)


[[-0.9115058  -0.71506697]
 [-0.31258115 -0.93307924]
 [-0.25014585  0.15666881]
 [-0.56317055 -0.83777404]]


NumericArray('log_prob', Array(-7.3515086, dtype=float32)) NumericArray('log_prob', Array(-6.9956217, dtype=float32))


## 4. Operations on laws

The operations form a registry, and `describe` lists an operation's operands and its routes in selection order.

In [14]:
print([summary.name for summary in operation_registry.list()])
print(operation_registry.describe("mean"))

['convert', 'sample', 'condition_on', 'log_prob', 'unnormalized_log_prob', 'prob', 'unnormalized_prob', 'random_log_prob', 'random_unnormalized_log_prob', 'evaluate', 'inverse', 'log_det_jacobian', 'joint', 'marginal', 'factor', 'mixture', 'mean', 'variance', 'cov', 'quantile', 'expectation']
mean(d) — primitive
  The mean ``E[X]`` for ``X ~ d``, a value shaped like one draw.
  operands:
    d: DistributionSpec; read by the result rule
  routes, in selection order:
    closed_form (capability, exact; requires SupportsMean): membership in SupportsMean suffices
    monte_carlo (fallback, approximate): The law samples, and the moment is assumed to exist.


`sample` returns one tracked draw, or a batch on a level named `sample`, and `with_options(raw=True)` returns the representation instead.
Draws inside `workflow_run(seed=...)` are reproducible (Section 12).
**Transitional:** `key=` arguments retire, and the scope's seed fixes the draws (S1).

In [15]:
mu_prior = Normal("mu", 0.0, 5.0)
with workflow_run(seed=0):
    one = sample(mu_prior)
    many = sample(mu_prior, sample_shape=(5,))
    raw_many = sample.with_options(raw=True)(mu_prior, sample_shape=(5,))
    prior_draws = sample(prior, sample_shape=(3,))
print(repr(one))
print(repr(many), many.values)
print(type(raw_many).__name__, raw_many)
print(repr(prior_draws))
print(repr(prior_draws[0]))

NumericArray('sample', Array(-4.557529, dtype=float32))
NumericArrayBatch(batch_shape=(5,), levels=('sample',), event_shape=()) [-5.127493  -1.9288195 -0.6630897  6.3674555  3.6620731]
ArrayImpl [-14.860659   -3.721889   -6.036863    3.994918    6.0235777]
NumericRecordBatch(name='sample', sample=3)
NumericRecord(population=NumericRecord(mu=Array(3.6239264, dtype=float32), tau=Array(8.899201, dtype=float32)), groups=NumericRecord(theta_tilde=array(shape=(8,))))


**Finding:** every result is labeled by the operation alone (`sample`, `log_prob`, `mean`), so two means of different laws share the label `mean`, where II.4 derives a result's name from its inputs.

The functionals return values of the event's kind.
`cov` is over the flattened event, and `quantile` at several levels returns a batch on a level named `quantile`, the level axis leading.

In [16]:
z = families[-1]
print(repr(log_prob(z, jnp.zeros(2))))
print(repr(mean(z)), repr(variance(z)))
print(repr(cov(z)))
levels = quantile(z, jnp.array([0.05, 0.5, 0.95]))
print(repr(levels))
print(levels.values)
print(repr(quantile(z, 0.5)))

NumericArray('log_prob', Array(-2.1176848, dtype=float32))


NumericArray('mean', Array([0., 0.], dtype=float32)) NumericArray('variance', Array([1., 2.], dtype=float32))
NumericArray('cov', Array([[1. , 0.5],
       [0.5, 2. ]], dtype=float32))


NumericArrayBatch(batch_shape=(3,), levels=('quantile',), event_shape=(2,))
[[-1.6448536 -2.3261743]
 [ 0.         0.       ]
 [ 1.6448536  2.3261743]]


NumericArray('quantile', Array([0., 0.], dtype=float32))


**Finding:** `cov` returns a dense `NumericArray`, while design VI.5 returns a `LinOp` over the flattened draw.

`expectation(d, f)` integrates `f`, and `evaluate(f, d)` returns the pushforward law, which for a nonlinear map is an empirical law over the outputs.
**Transitional:** `expectation` resolves through its own method registry, whose methods are `exact` and `monte_carlo`. It becomes `mean(evaluate(f, d))`, with `evaluate`'s controls and method names (Z8).

In [17]:
@function
def square(x):
    return x**2


with workflow_run(seed=0):
    second_moment = expectation(mu_prior, square)
    pushforward = evaluate(square, mu_prior)
print(repr(second_moment), "(exact: 25)")
print(repr(pushforward), pushforward.event_spec.spec)
print(pushforward.provenance.metadata)
print(repr(mean(pushforward)))

NumericArray('expectation', Array(23.685886, dtype=float32)) (exact: 25)
EmpiricalDistribution(name='evaluate', num_atoms=128) NumericArraySpec(shape=(), dtype=dtype('float32'), support=None)
{'func': 'evaluate', 'route': 'evaluation_rules', 'exact': False}
NumericArray('mean', Array(29.23074, dtype=float32))


`check` runs the call up to the selection of a route without executing it, and returns a `CallReport`: the routes in selection order, the selected one, and the planned result.

In [18]:
report = mean.check(z)
print(report)
show_report(report)
show_report(expectation.check(mu_prior, square))

CallReport(routes=(MethodInfo(feasible=True, description='', pending=(), method_name='closed_form', exact=False),), selected=MethodInfo(feasible=True, description='', pending=(), method_name='closed_form', exact=True), deferred=(), result=OutputSpec(_component_name='z', _term_spec=NumericArraySpec(shape=(2,), dtype=dtype('float32'), support=real)), lifted=(), conversions=mappingproxy({}))
  closed_form                        feasible  exact=False  
selected: closed_form | exact: True
  methods (exact methods)            declined  exact=False  No feasible method for Normal with exact_only. Tried: exact: Normal has no exact expectation
  methods (approximate methods)      feasible  exact=False  
selected: methods/monte_carlo | exact: False


**Known bug:** `check` lists capability routes as approximate, so the `closed_form` route reads `exact=False` among the routes and `exact=True` once selected.

## 5. The hierarchical model three ways

The non-centered eight-schools model has `mu ~ N(0, 5)`, `tau ~ HalfCauchy(0, 5)`, `theta_tilde_j ~ N(0, 1)`, and `y_j ~ N(mu + tau theta_tilde_j, sigma_j)`.
The three representations below are conditioned on the observed effects, and each posterior ends with the parameter record `{population: {mu, tau}, groups: {theta_tilde}}`.

### Stan

`StanModel` is a `ConditionalDistribution` from the program's data block to the unnormalized posterior over its parameters.
The adapter reads both declarations from the program before any data are bound, and `J` stays a symbolic dimension of `theta_tilde`.

In [19]:
STAN_PROGRAM = '''
data {
  int<lower=1> J;
  vector[J] y;
  vector<lower=0>[J] sigma;
}
parameters {
  real mu;
  real<lower=0> tau;
  vector[J] theta_tilde;
}
model {
  mu ~ normal(0, 5);
  tau ~ cauchy(0, 5);
  theta_tilde ~ normal(0, 1);
  y ~ normal(mu + tau * theta_tilde, sigma);
}
'''
stan_file = Path(tempfile.mkdtemp()) / "eight_schools.stan"
stan_file.write_text(STAN_PROGRAM)
stan_model = StanModel("eight_schools", str(stan_file))
stan_data = {"J": J, "y": np.asarray(y_obs), "sigma": np.asarray(sigma)}
print(type(stan_model).__name__, stan_model.name)
print("given:", dict(stan_model.given_spec))
print("event:", stan_model.event_spec.spec)

StanModel eight_schools
given: {'J': OpaqueSpec(meta=None), 'y': OpaqueSpec(meta=None), 'sigma': OpaqueSpec(meta=None)}
event: NumericRecordSpec(mu=(), tau=(), theta_tilde=('J',))


**Finding:** the given slots are typed `OpaqueSpec`, while design VII.9 gives each slot the numeric spec of its declared type.

Binding the data curries the kernel to its unnormalized posterior, and `condition_on` normalizes it with nutpie, the highest-ranked method that applies.
The posterior is empirical over the draws, and renaming it nests its fields.

In [20]:
show_report(condition_on.check(stan_model, stan_data))
NUTS = {"num_warmup": 500, "num_results": 500, "num_chains": 2}
with workflow_run(seed=1):
    stan_fit = condition_on.with_options(method_options=NUTS)(stan_model, stan_data)
stan_posterior = stan_fit.with_path_names(NEST)
print(repr(stan_fit), stan_fit.event_spec.spec)
print(repr(stan_posterior), stan_posterior.event_spec.spec)

  curry (exact methods)              declined  exact=False  the exact stage's result is unnormalized, and no exact method normalizes it; method="unnormalized" returns that result: No method registered for _StanPosterior with exact_only. Available: ['nutpie_nuts', 'blackjax_nuts', 'cmdstan_nuts', 'pymc_nuts', 'blackjax_elliptical_slice', 'blackjax_rwmh', 'blackjax_sgld', 'tfp_nuts', 'tfp_hmc', 'blackjax_hmc', 'blackjax_sghmc', 'pymc_advi']
  slice (exact methods)              declined  exact=False  route 'slice' declined: the conditioned object is not a joint law
  exact_conditioning                 declined  exact=False  StanModel does not claim SupportsExactConditioning
  bayes (exact methods)              declined  exact=False  route 'bayes' declined: the given names no produced field
  curry (approximate methods)        feasible  exact=False  
selected: curry/nutpie_nuts | exact: False


Progress,Draws,Divergences,Step Size,Gradients/Draw
,1000,0,0.68,7
,1000,0,0.69,7


ApproximateDistribution(algorithm='workflow.condition_on', num_chains=2, num_draws=500, components=['mu', 'tau', 'theta_tilde']) NumericRecordSpec(mu=(), tau=(), theta_tilde=(8,))
_RenamedDistribution(name='condition_on', parent='condition_on') NumericRecordSpec(population=NumericRecordSpec(mu=(), tau=()), groups=NumericRecordSpec(theta_tilde=(8,)))


**Finding:** renaming the `StanModel` before conditioning moves its target off nutpie's path.
The registry selects `blackjax_rwmh`, whose chain starts at a negative `tau`, and BridgeStan refuses the point.
This looks like the known bug of chains that start outside the support, on another method.

In [21]:
renamed_stan = stan_model.with_path_names(NEST)
print(condition_on.check(renamed_stan, stan_data).selected.method_name)
try:
    condition_on.with_options(method_options={"num_warmup": 50, "num_results": 50, "num_chains": 1})(renamed_stan, stan_data)
except RuntimeError as error:
    print(f"{type(error).__name__}: {str(error)[:160]}")

curry/blackjax_rwmh


RuntimeError: param_unconstrain() failed with exception: Exception: lb_free: Lower bounded variable is -0.0832902, but must be greater than or equal to 0.000000 (in '/private


### PyMC

`PyMCModel` is the joint law that a model-building function defines over its free variables.
An argument that defaults to `None` and feeds an observed variable is an event field, so this model is a `Distribution` over `{mu, tau, theta_tilde, y}`.

In [22]:
def eight_schools_pymc(y=None):
    import pymc as pm

    with pm.Model() as model:
        mu = pm.Normal("mu", 0.0, 5.0)
        tau = pm.HalfCauchy("tau", 5.0)
        theta_tilde = pm.Normal("theta_tilde", 0.0, 1.0, shape=J)
        pm.Normal("y", mu + tau * theta_tilde, np.asarray(sigma), observed=y, shape=J)
    return model


pymc_model = PyMCModel("eight_schools", eight_schools_pymc)
print(repr(pymc_model), pymc_model.event_spec.spec)
show_report(condition_on.check(pymc_model, {"y": np.asarray(y_obs)}))
with workflow_run(seed=1):
    pymc_fit = condition_on.with_options(method_options=NUTS)(pymc_model, {"y": np.asarray(y_obs)})
pymc_posterior = pymc_fit.with_path_names(NEST)
print(repr(pymc_posterior), pymc_posterior.event_spec.spec)

PyMCModel(variables=[mu, tau, theta_tilde, y]) NumericRecordSpec(mu=(), tau=(), theta_tilde=(8,), y=(8,))
  curry (exact methods)              declined  exact=False  route 'curry' declined: the conditioned object is not a kernel
  slice (exact methods)              declined  exact=False  route 'slice' declined: the conditioned object is not a joint law
  exact_conditioning                 declined  exact=False  PyMCModel does not claim SupportsExactConditioning
  bayes (exact methods)              declined  exact=False  the exact stage's result is unnormalized, and no exact method normalizes it; method="unnormalized" returns that result: No method registered for _UnnormalizedConditional with exact_only. Available: ['nutpie_nuts', 'blackjax_nuts', 'cmdstan_nuts', 'pymc_nuts', 'blackjax_elliptical_slice', 'blackjax_rwmh', 'blackjax_sgld', 'tfp_nuts', 'tfp_hmc', 'blackjax_hmc', 'blackjax_sghmc', 'pymc_advi']
  curry (approximate methods)        declined  exact=False  route 'curry' decline

Progress,Draws,Divergences,Step Size,Gradients/Draw
,1000,0,0.51,7
,1000,0,0.57,3


_RenamedDistribution(name='condition_on', parent='condition_on') NumericRecordSpec(population=NumericRecordSpec(mu=(), tau=()), groups=NumericRecordSpec(theta_tilde=(8,)))


**Finding:** renaming the PyMC model before conditioning also leaves nutpie, here for `blackjax_rwmh`, a random-walk sampler.

In [23]:
print(condition_on.check(pymc_model.with_path_names(NEST), {"y": np.asarray(y_obs)}).selected.method_name)

bayes/blackjax_rwmh


### A native factored joint

A native likelihood is a `ConditionalDistribution` subclass.
Its given slots are the prior's two groups, and `_condition_on` returns the law of `y` at one given value.
The class also implements the conditional sampler and density, so the joint samples and scores.

In [24]:
def raw_fields(value):
    # A given arrives as a Record or as a mapping.
    return value.to_nested_dict() if isinstance(value, Record) else dict(value)


class SchoolEffects(ConditionalDistribution, SupportsConditionalSampling, SupportsConditionalLogProb):
    # y_j | population, groups ~ N(mu + tau theta_tilde_j, sigma_j)

    def __init__(self, name, sigma):
        record = prior.event_spec.spec
        super().__init__(
            name,
            {"population": record.at_path("population"), "groups": record.at_path("groups")},
            NumericArraySpec((J,), jnp.float32, real),
        )
        self.sigma = sigma

    def _law(self, given):
        given = raw_fields(given)
        population, groups = raw_fields(given["population"]), raw_fields(given["groups"])
        return Normal(self.name, population["mu"] + population["tau"] * groups["theta_tilde"], self.sigma)

    def _condition_on(self, given, /, **options):
        return self._law(given)

    def _conditional_sample(self, given, key, sample_shape=()):
        return self._law(given)._sample(key, sample_shape)

    def _conditional_log_prob(self, given, value):
        return self._law(given)._log_prob(value)


likelihood = SchoolEffects("y", sigma)
schools = likelihood * prior
print(repr(likelihood), list(likelihood.given_spec), likelihood.event_spec.spec)
print(repr(schools), schools.event_spec.spec)

SchoolEffects(name='y') ['population', 'groups'] NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)
FactoredDistribution(name='y·mu·tau·theta_tilde') NumericRecordSpec(y=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real), population=NumericRecordSpec(mu=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), tau=NumericArraySpec(shape=(), dtype=dtype('float32'), support=greater_than(0.0))), groups=NumericRecordSpec(theta_tilde=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)))


Composition is conditional-first: the likelihood conditions on what the prior produces, so `likelihood * prior` is a `FactoredDistribution` over `{y, population, groups}`, labeled by joining the operands' labels with `·`.
It samples ancestrally and scores a draw.

In [25]:
with workflow_run(seed=0):
    joint_draw = sample(schools)
print(repr(joint_draw))
print(repr(log_prob(schools, joint_draw)))

NumericRecord(y=array(shape=(8,)), population=NumericRecord(mu=Array(1.4323593, dtype=float32), tau=Array(3.3399086, dtype=float32)), groups=NumericRecord(theta_tilde=array(shape=(8,))))
NumericArray('log_prob', Array(-44.87911, dtype=float32))


Conditioning on `y` resolves to Bayes' rule, and the inference-method registry selects BlackJAX's NUTS; TFP's NUTS runs when named.

In [26]:
show_report(condition_on.check(schools, {"y": y_obs}))
with workflow_run(seed=1):
    native_posterior = condition_on.with_options(method_options=NUTS)(schools, {"y": y_obs})
    tfp_posterior = condition_on.with_options(method="tfp_nuts", method_options=NUTS)(schools, {"y": y_obs})
print(repr(native_posterior))
print(repr(tfp_posterior))

  curry (exact methods)              declined  exact=False  route 'curry' declined: the conditioned object is not a kernel
  slice (exact methods)              declined  exact=False  route 'slice' declined: the factor producing ['y'] conditions on ['groups', 'population'], which the given leaves free, so the conditional is not a product of the factors
  exact_conditioning                 declined  exact=False  FactoredDistribution does not claim SupportsExactConditioning
  bayes (exact methods)              declined  exact=False  the exact stage's result is unnormalized, and no exact method normalizes it; method="unnormalized" returns that result: No method registered for _UnnormalizedConditional with exact_only. Available: ['nutpie_nuts', 'blackjax_nuts', 'cmdstan_nuts', 'pymc_nuts', 'blackjax_elliptical_slice', 'blackjax_rwmh', 'blackjax_sgld', 'tfp_nuts', 'tfp_hmc', 'blackjax_hmc', 'blackjax_sghmc', 'pymc_advi']
  curry (approximate methods)        declined  exact=False  route 'curr

ApproximateDistribution(algorithm='workflow.condition_on', num_chains=2, num_draws=500, components=['population', 'groups'])
ApproximateDistribution(algorithm='workflow.condition_on', num_chains=2, num_draws=500, components=['population', 'groups'])


### Comparison

The school effects are `theta_j = mu + tau theta_tilde_j`, a function of three fields of the posterior.
Passing the posterior's field views lifts the function, and the views co-sample, so each evaluation reads one posterior draw.
With `n_broadcast_samples` at the atom count, the lift enumerates an empirical posterior exactly.

In [27]:
@function
def school_effects(mu, tau, theta_tilde):
    return mu + tau * theta_tilde


posteriors = {
    "Stan, nutpie": stan_posterior,
    "PyMC, nutpie": pymc_posterior,
    "native, BlackJAX NUTS": native_posterior,
    "native, TFP NUTS": tfp_posterior,
}
rows = {}
for label, post in posteriors.items():
    with workflow_run(seed=4):
        thetas = school_effects.with_options(n_broadcast_samples=1000)(
            post["population/mu"], post["population/tau"], post["groups/theta_tilde"]
        )
    means, variances = mean(post), variance(post)
    summaries = [
        ("mu", means["population/mu"], variances["population/mu"]),
        ("tau", means["population/tau"], variances["population/tau"]),
        *[(f"theta[{j}]", mean(thetas).value[j], variance(thetas).value[j]) for j in range(3)],
    ]
    rows[label] = {(name, stat): value for name, m, v in summaries for stat, value in (("mean", float(m)), ("sd", float(v) ** 0.5))}
    rows[label][("lift", "route")] = thetas.provenance.metadata["route"]
pd.DataFrame.from_dict(rows, orient="index").round(2)

mu         tau       theta[0]       theta[1]        \
                       mean    sd  mean    sd     mean    sd     mean    sd   
Stan, nutpie           4.50  3.46  3.48  3.10     6.63  5.41     5.31  4.65   
PyMC, nutpie           4.38  3.24  3.61  3.15     6.27  6.07     4.78  4.25   
native, BlackJAX NUTS  4.38  3.24  3.65  3.14     6.16  5.62     5.09  4.70   
native, TFP NUTS       4.86  3.41  3.52  3.22     6.68  5.65     5.55  4.61   

                      theta[2]                         lift  
                          mean    sd                  route  
Stan, nutpie              4.41  5.24          sampling_lift  
PyMC, nutpie              3.80  4.85          sampling_lift  
native, BlackJAX NUTS     4.18  5.14  empirical_enumeration  
native, TFP NUTS          4.51  5.18  empirical_enumeration

**Finding:** the renamed posteriors take the sampling lift, since a renamed empirical law is no longer an empirical law, and the enumeration declines.

Each result declares its target's event, with the Stan dimension `J` bound.
Its provenance records the route, the method, and whether the step was exact.

In [28]:
print("native target:   ", prior.event_spec.spec)
print("native posterior:", native_posterior.event_spec.spec)
print("Stan target:     ", stan_model.event_spec.spec)
print("Stan posterior:  ", stan_fit.event_spec.spec)
print(native_posterior.provenance)
print(native_posterior.provenance.metadata)
print(stan_fit.provenance.metadata)

native target:    NumericRecordSpec(population=NumericRecordSpec(mu=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), tau=NumericArraySpec(shape=(), dtype=dtype('float32'), support=greater_than(0.0))), groups=NumericRecordSpec(theta_tilde=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)))
native posterior: NumericRecordSpec(population=NumericRecordSpec(mu=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), tau=NumericArraySpec(shape=(), dtype=dtype('float32'), support=greater_than(0.0))), groups=NumericRecordSpec(theta_tilde=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real)))
Stan target:      NumericRecordSpec(mu=(), tau=(), theta_tilde=('J',))
Stan posterior:   NumericRecordSpec(mu=(), tau=(), theta_tilde=(8,))
Provenance('workflow.condition_on', parents=[condition_on, y·mu·tau·theta_tilde, posterior])
{'func': 'condition_on', 'route': 'bayes', 'exact': False, 'method': 'blackjax_nuts'}
{'func': 'condition_on', 'route':

**Finding:** the program-defined events declare shapes alone, with no dtype and no support, so `tau`'s lower bound is not declared, while the native posterior declares both.
The posterior's repr reads `algorithm='workflow.condition_on'`, so the method shows only in the provenance metadata.

`exact_only=True` excludes every approximate route.
No exact method normalizes the unnormalized posterior, so the call raises, and the message names `method="unnormalized"`.

In [29]:
try:
    condition_on.with_options(exact_only=True)(stan_model, stan_data)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")

ResolutionError: condition_on: no route applies with exact_only. Tried: curry (exact methods): the exact stage's result is unnormalized, and no exact method normalizes it; method="unnormalized" returns that result: No method registered for _StanPosterior with exact_only. Available: ['nutpie_nuts', 'blackjax_nuts', 'cmdstan_nuts', 'pymc_nuts', 'blackjax_elliptical_slice', 'blackjax_rwmh', 'blackjax_sgld', 'tfp_nuts', 'tfp_hmc', 'blackjax_hmc', 'blackjax_sghmc', 'pymc_advi']; slice (exact methods): route 'slice' declined: the conditioned object is not a joint law; exact_conditioning: StanModel does not claim SupportsExactConditioning; bayes (exact methods): route 'bayes' declined: the given names no produced field


`method="unnormalized"` returns the exact stage alone: the unnormalized posterior, which claims only `SupportsUnnormalizedLogProb`.

In [30]:
stan_target = condition_on.with_options(method="unnormalized")(stan_model, stan_data)
native_target = condition_on.with_options(method="unnormalized")(schools, {"y": y_obs})
claims = (SupportsSampling, SupportsLogProb, SupportsUnnormalizedLogProb)
for target in (stan_target, native_target):
    print(type(target).__name__, [p.__name__ for p in claims if isinstance(target, p)])
print(repr(unnormalized_log_prob(stan_target, {"mu": 4.0, "tau": 3.0, "theta_tilde": np.zeros(J)})))
print(repr(unnormalized_log_prob(native_target, {"population": {"mu": 4.0, "tau": 3.0}, "groups": {"theta_tilde": jnp.zeros(J)}})))

_StanPosterior ['SupportsUnnormalizedLogProb']
_UnnormalizedConditional ['SupportsUnnormalizedLogProb']


NumericArray('unnormalized_log_prob', Array(-3.3906543, dtype=float32, weak_type=True))
NumericArray('unnormalized_log_prob', Array(-42.652267, dtype=float32))


Conditioning on the upstream field `population` would be an exact slice of a factored prior, `p(y | population, groups) p(groups)`.
The renamed prior is not factored, so the call resolves to Bayes' rule with NUTS over `y` and `groups`.
A slice would also call `SchoolEffects._condition_on` with a partial given, which the class above does not curry.

In [31]:
show_report(condition_on.check(schools, {"population": {"mu": 4.0, "tau": 3.0}}))

  curry (exact methods)              declined  exact=False  route 'curry' declined: the conditioned object is not a kernel
  slice (exact methods)              declined  exact=False  route 'slice' declined: the factor producing ['population'] produces ['groups'] as well, and it does not condition on them exactly, since its fields are not assumed independent
  exact_conditioning                 declined  exact=False  FactoredDistribution does not claim SupportsExactConditioning
  bayes (exact methods)              declined  exact=False  the exact stage's result is unnormalized, and no exact method normalizes it; method="unnormalized" returns that result: No method registered for _UnnormalizedConditional with exact_only. Available: ['nutpie_nuts', 'blackjax_nuts', 'cmdstan_nuts', 'pymc_nuts', 'blackjax_elliptical_slice', 'blackjax_rwmh', 'blackjax_sgld', 'tfp_nuts', 'tfp_hmc', 'blackjax_hmc', 'blackjax_sghmc', 'pymc_advi']
  curry (approximate methods)        declined  exact=False  route

## 6. Field views and marginals

`d[path]` is a field view: a law over the field at `path` that refers to its parent.
A view of a leaf works on the joint and on the posterior.
**Transitional:** a view is labeled by its path, and it keeps its parent's label once the label fix lands (Z5).

In [32]:
tau_view = schools["population/tau"]
print(repr(tau_view), "| label:", tau_view.name, "| event:", tau_view.event_spec.spec)
print(repr(native_posterior["groups/theta_tilde"]))
print(repr(mean(native_posterior["groups/theta_tilde"])))

FieldView(parent='y·mu·tau·theta_tilde', path='population/tau') | label: population/tau | event: NumericArraySpec(shape=(), dtype=dtype('float32'), support=greater_than(0.0))
FieldView(parent='condition_on', path='groups/theta_tilde')
NumericArray('mean', Array([ 0.28817454,  0.1414243 ,  0.01084518,  0.03623313, -0.15173559,
       -0.07823739,  0.39439055,  0.13329877], dtype=float32))


**Known bug:** `d[path]` at a regrouped node raises, so `population`, which the rename gathered from two fields, has no view.

In [33]:
try:
    schools["population"]
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: 'population' holds no single node of 'mu·tau·theta_tilde', since a move gathered or regrouped its fields


A view derives its capabilities from its parent's.
The view of `y` claims `SupportsLogProb`, but its guard declines, since the marginal density of `y` integrates out the parameters that `y` conditions on.

In [34]:
y_view = schools["y"]
print(isinstance(y_view, SupportsLogProb))
try:
    log_prob(y_view, y_obs)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")
print(repr(log_prob(tau_view, 1.0)))

True
ResolutionError: log_prob: no route applies. Tried: exact: the marginal integrates out ['mu·tau·theta_tilde'], which the requested fields condition on
NumericArray('log_prob', Array(-2.1002414, dtype=float32))


**Finding:** the message names the factor label `mu·tau·theta_tilde` rather than the fields `population` and `groups`.
**Open question:** a public capability for the density of a subset of observations, such as one school's `y_j`.

Views of one parent co-sample when lifted together, since the lift groups arguments by their root and draws the parent once per repetition.
Detached marginals draw independently.
Passing one field twice makes the difference visible.

In [35]:
@function
def difference(a, b):
    return a - b


with workflow_run(seed=0):
    from_views = difference(schools["population/mu"], schools["population/mu"])
    from_marginals = difference(marginal(schools, "population/mu"), marginal(schools, "population/mu"))
print(repr(from_views), "max |a - b| =", float(jnp.abs(from_views.atoms.values).max()))
print(repr(from_marginals), "max |a - b| =", float(jnp.abs(from_marginals.atoms.values).max()))

EmpiricalDistribution(name='difference', num_atoms=128) max |a - b| = 0.0
EmpiricalDistribution(name='difference', num_atoms=128) max |a - b| = 21.93586540222168


`marginal` returns the detached marginal.
A root field's marginal is its exact factor, while a field that the likelihood produces has no exact route.
**Transitional:** the marginal is labeled `marginal`, and it keeps its parent's label after the fix (Z5).

In [36]:
mu_marginal = marginal(schools, "population/mu")
print(repr(mu_marginal), mu_marginal.event_spec.spec, mu_marginal.provenance.metadata)
show_report(marginal.check(schools, "y"))
try:
    marginal(schools, "y")
except NotImplementedError as error:
    print(f"{type(error).__name__}: {error}")

Normal(name='marginal', event_shape=()) NumericArraySpec(shape=(), dtype=dtype('float32'), support=real) {'func': 'marginal', 'route': 'exact', 'exact': True}
  exact                              declined  exact=False  the marginal integrates out ['mu·tau·theta_tilde'], which the requested fields condition on
  monte_carlo                        feasible  exact=False  
selected: monte_carlo | exact: False
NotImplementedError: marginal.monte_carlo


**Finding:** `check` selects the `monte_carlo` route for `y`, and its execution raises `NotImplementedError`; `tests/correctness/test_models.py` marks the case pending.

On the empirical posterior every marginal is exact, a selection included, and the selection's fields follow the order of its paths.

In [37]:
pair = marginal(native_posterior, ("groups/theta_tilde", "population/mu"))
print(repr(pair), pair.event_spec.spec)
print(repr(native_posterior[("groups/theta_tilde", "population/mu")]))

EmpiricalDistribution(name='marginal', num_atoms=1000) NumericRecordSpec(theta_tilde=NumericArraySpec(shape=(8,), dtype=dtype('float32'), support=real), mu=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real))
FieldView(parent='condition_on', path=('groups/theta_tilde', 'population/mu'))


**Known bug:** on a factored joint, a selection's marginal orders its fields by factor, while the view keeps the requested order.

In [38]:
ab = Normal("a", 0.0, 1.0) * Normal("b", 5.0, 1.0)
print("view:    ", ab[("b", "a")].event_spec.spec)
print("marginal:", marginal(ab, ("b", "a")).event_spec.spec)

view:     NumericRecordSpec(b=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), a=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real))
marginal: NumericRecordSpec(a=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real), b=NumericArraySpec(shape=(), dtype=dtype('float32'), support=real))


**Finding:** a view and a marginal of one field answer a moment differently.
The joint is dependent, so the view's mean takes the Monte Carlo route and returns a number for `tau`, whose mean does not exist, while the exact marginal raises `MathematicalDomainError`.

In [39]:
with workflow_run(seed=0):
    print(repr(mean(tau_view)), "via", mean.check(tau_view).selected.method_name)
try:
    mean(marginal(schools, "population/tau"))
except MathematicalDomainError as error:
    print(f"{type(error).__name__}: {error}")

NumericArray('mean', Array(31.365866, dtype=float32)) via monte_carlo
MathematicalDomainError: the mean of 'marginal' does not exist: E[X] is infinite


`factor` returns the building block that produces a component: the likelihood for `y`, and the renamed prior for `groups`.

In [40]:
print(repr(factor(schools, component_name="y")))
print(repr(factor(schools, component_name="groups")))

SchoolEffects(name='factor')
_RenamedDistribution(name='factor', parent='mu·tau·theta_tilde')
